# 02 - 单指标研究框架（Triple Barrier Method）

## 核心思路

趋势预测指标的评估不能简单用 `corr(indicator, future_N_bar_return)`，
因为趋势的持续时长未知。

**Triple Barrier Method 路径依赖标签**：
- 上轨（如 +2%）→ 标签 1（强劲上涨）
- 下轨（如 -1.5%）→ 标签 -1（强劲下跌）
- 时间屏障 → 标签 0（震荡）

模型学习：给定指标值时，"未来出现显著趋势行情的概率"。

## 流程
1. 加载 aggTrades → resample OHLCV
2. 计算单个指标
3. 生成 Triple Barrier 标签
4. 评估指标在不同标签下的分布差异
5. 回测指标的信号质量

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

from utilities.binance_loader import load_agg_trades, resample_trades_to_ohlcv
from backtest.labeling import triple_barrier_labels, multi_horizon_labels
from backtest.engine import backtest_single_indicator

# 指标库
from indicators.base.price_impact import tick_price_impact, volume_weighted_impact, kyles_lambda
from indicators.base.spread import trade_diff_spread, roll_spread, spread_percentile
from indicators.base.order_flow import trade_imbalance, vpin, flow_toxicity
from indicators.base.volume_profile import (
    taker_volume_corr, taker_volume_autocorr, taker_volume_skewness,
    buy_sell_volume_ratio, large_trade_ratio,
)

# Regime
from indicators.regime.volatility_regime import realized_volatility, parkinson_volatility, volatility_regime
from indicators.regime.trend_strength import adx_indicator, efficiency_ratio, hurst_exponent
from indicators.regime.liquidity_regime import amihud_illiquidity

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

## Step 1: 数据加载

In [ ]:
SYMBOL = 'SOL/USDT'
DAYS = 3

trades = load_agg_trades(symbol=SYMBOL, days=DAYS)
print(f'aggTrades: {len(trades):,}')

# Resample 为不同频率
ohlcv_1m = resample_trades_to_ohlcv(trades, freq='1min')
ohlcv_5m = resample_trades_to_ohlcv(trades, freq='5min')
print(f'1min bars: {len(ohlcv_1m)},  5min bars: {len(ohlcv_5m)}')

## Step 2: 计算所有基础指标

将指标统一 resample 到 OHLCV 的频率（1min / 5min）以便分析。

In [ ]:
# ── Tick-level 指标 ──
print('Computing tick-level indicators...')
ind_impact = tick_price_impact(trades, window_ms=10000)
ind_vw_impact = volume_weighted_impact(trades, window_ms=10000)
ind_lambda = kyles_lambda(trades, window_trades=200)
ind_spread = trade_diff_spread(trades, window_trades=200)
ind_roll = roll_spread(trades, window_trades=500)
ind_imbalance = trade_imbalance(trades, window_trades=200)
ind_toxicity = flow_toxicity(trades, window_trades=200)

# ── Taker-level 指标（Binance aggTrades 专属）──
print('Computing taker-level indicators...')
ind_corr = taker_volume_corr(trades, resample_freq='5s', window=120)
ind_autocorr_buy = taker_volume_autocorr(trades, resample_freq='5s', window=120, side='buy')
ind_autocorr_sell = taker_volume_autocorr(trades, resample_freq='5s', window=120, side='sell')
ind_skew = taker_volume_skewness(trades, resample_freq='5s', window=240)
ind_bsratio = buy_sell_volume_ratio(trades, resample_freq='5s', window=120)
ind_large = large_trade_ratio(trades, resample_freq='5s', window=120)

print('Done.')

In [ ]:
# 将 tick-level 指标 resample 到 1min（取最后一个值）
def resample_indicator(ind: pd.Series, freq: str = '1min') -> pd.Series:
    """将高频指标 resample 到目标频率。"""
    s = ind.copy()
    s = s[~s.index.duplicated(keep='last')]
    return s.resample(freq).last().dropna()

indicators_1m = pd.DataFrame({
    'price_impact':    resample_indicator(ind_impact),
    'vw_impact':       resample_indicator(ind_vw_impact),
    'kyles_lambda':    resample_indicator(ind_lambda),
    'spread':          resample_indicator(ind_spread),
    'roll_spread':     resample_indicator(ind_roll),
    'trade_imbalance': resample_indicator(ind_imbalance),
    'flow_toxicity':   resample_indicator(ind_toxicity),
    'taker_vol_corr':  resample_indicator(ind_corr),
    'autocorr_buy':    resample_indicator(ind_autocorr_buy),
    'autocorr_sell':   resample_indicator(ind_autocorr_sell),
    'taker_skew':      resample_indicator(ind_skew),
    'bs_ratio':        resample_indicator(ind_bsratio),
    'large_trade_pct': resample_indicator(ind_large),
})

# 对齐到 ohlcv
common_idx = indicators_1m.index.intersection(ohlcv_1m.index)
indicators_1m = indicators_1m.loc[common_idx]
prices_1m = ohlcv_1m.loc[common_idx, 'close']

print(f'Aligned: {len(common_idx)} bars, {len(indicators_1m.columns)} indicators')
indicators_1m.describe().round(4)

## Step 3: Triple Barrier 标签

生成不同参数的标签，研究指标在不同 regime 下的预测力。

In [ ]:
# 默认参数：上轨 +0.5%, 下轨 -0.3%, 窗口 60 bars（1h on 1min data）
tb_labels = triple_barrier_labels(
    prices_1m,
    upper_pct=0.5,
    lower_pct=0.3,
    max_bars=60,
)

print('标签分布：')
print(tb_labels['label'].value_counts())
print(f'\n上涨占比: {(tb_labels["label"]==1).mean():.1%}')
print(f'下跌占比: {(tb_labels["label"]==-1).mean():.1%}')
print(f'震荡占比: {(tb_labels["label"]==0).mean():.1%}')

In [ ]:
# 多 horizon 标签
multi_tb = multi_horizon_labels(
    prices_1m,
    horizons=[10, 30, 60, 120],
    upper_pct=0.5,
    lower_pct=0.3,
)

for h in [10, 30, 60, 120]:
    col = f'label_{h}'
    dist = multi_tb[col].value_counts().to_dict()
    print(f'Horizon {h:>3d} bars: Up={dist.get(1,0)}, Down={dist.get(-1,0)}, Flat={dist.get(0,0)}')

## Step 4: 指标 vs 标签分析

核心问题：指标在不同 TB 标签下的分布是否有显著差异？

In [ ]:
# 合并指标和标签
analysis = indicators_1m.copy()
analysis['tb_label'] = tb_labels.loc[common_idx, 'label']
analysis = analysis.dropna(subset=['tb_label'])

# 各指标在不同标签下的均值比较
group_means = analysis.groupby('tb_label').mean()
print('各指标在不同标签下的均值：')
group_means.T.round(6)

In [ ]:
# 可视化：指标分布 by label
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()

for i, col in enumerate(indicators_1m.columns):
    if i >= len(axes):
        break
    ax = axes[i]
    for label, color, name in [(1, 'green', 'Up'), (-1, 'red', 'Down'), (0, 'gray', 'Flat')]:
        subset = analysis[analysis['tb_label'] == label][col].dropna()
        if len(subset) > 0:
            ax.hist(subset, bins=50, alpha=0.4, color=color, label=name, density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

# Hide unused axes
for j in range(len(indicators_1m.columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('指标分布 by Triple Barrier Label', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 统计检验：Kruskal-Wallis H-test（非参数）
from scipy import stats

results_test = []
for col in indicators_1m.columns:
    groups = [analysis[analysis['tb_label'] == lbl][col].dropna() for lbl in [1, -1, 0]]
    groups = [g for g in groups if len(g) > 5]
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        results_test.append({'indicator': col, 'H_stat': stat, 'p_value': p})

test_df = pd.DataFrame(results_test).sort_values('p_value')
test_df['significant'] = test_df['p_value'] < 0.05
print('Kruskal-Wallis 检验（指标在不同标签下的分布差异）：')
test_df

## Step 5: 单指标回测

对每个指标执行标准化回测，评估信号质量。

In [ ]:
# 参数
BT_PARAMS = dict(
    hold_bars=30,
    stop_loss_pct=0.3,
    take_profit_pct=0.5,
    cooldown_bars=5,
    use_triple_barrier=True,
    tb_upper_pct=0.5,
    tb_lower_pct=0.3,
    tb_max_bars=60,
)

# 定义每个指标的信号阈值
indicator_configs = {
    'trade_imbalance': {'long_threshold': 0.3, 'short_threshold': -0.3},
    'bs_ratio':        {'long_threshold': 0.5, 'short_threshold': -0.5},
    'flow_toxicity':   {'long_threshold': 5.0, 'short_threshold': -5.0},
    'taker_vol_corr':  {'long_threshold': 0.5, 'short_threshold': -0.5},
    'taker_skew':      {'long_threshold': 2.0, 'short_threshold': -2.0},
    'large_trade_pct': {'long_threshold': 0.5, 'short_threshold': -0.5},
}

bt_results = []
for name, config in indicator_configs.items():
    if name not in indicators_1m.columns:
        continue
    ind = indicators_1m[name].dropna()
    result = backtest_single_indicator(
        indicator=ind,
        prices=prices_1m,
        **config,
        **BT_PARAMS,
    )
    m = result['metrics']
    m['indicator'] = name
    m['n_signals'] = result.get('n_signals', 0)
    bt_results.append(m)
    print(f'{name:>20s}: trades={m["n_signals"]:>4d}  win={m["win_rate"]:>5.1f}%  '
          f'sharpe={m["sharpe"]:>6.2f}  avg_ret={m["avg_return"]:>7.4f}%')

bt_summary = pd.DataFrame(bt_results).set_index('indicator')
bt_summary

In [ ]:
# 可视化：指标回测比较
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

if not bt_summary.empty:
    bt_summary['sharpe'].plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('Sharpe Ratio')
    axes[0].axvline(0, color='black', linewidth=0.5)

    bt_summary['win_rate'].plot.barh(ax=axes[1], color='green')
    axes[1].set_title('Win Rate (%)')
    axes[1].axvline(50, color='red', linewidth=0.5, linestyle='--')

    bt_summary['profit_factor'].plot.barh(ax=axes[2], color='orange')
    axes[2].set_title('Profit Factor')
    axes[2].axvline(1, color='red', linewidth=0.5, linestyle='--')

plt.suptitle('单指标回测对比', fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: 最佳指标深入分析

选取 Sharpe 最高的指标进行详细分析。

In [ ]:
if not bt_summary.empty:
    best = bt_summary['sharpe'].idxmax()
    print(f'Best indicator: {best}')
    
    # 详细回测
    config = indicator_configs.get(best, {'long_threshold': 0, 'short_threshold': 0})
    detail = backtest_single_indicator(
        indicator=indicators_1m[best].dropna(),
        prices=prices_1m,
        **config,
        **BT_PARAMS,
    )
    
    rdf = detail['results_df']
    if not rdf.empty:
        # 累计收益曲线
        fig, ax = plt.subplots(figsize=(14, 4))
        rdf['cum_ret'] = rdf['return_pct'].cumsum()
        ax.plot(rdf['entry_time'], rdf['cum_ret'], 'b-', linewidth=1)
        ax.fill_between(rdf['entry_time'], 0, rdf['cum_ret'], alpha=0.2)
        ax.set_title(f'{best} - 累计收益曲线')
        ax.set_ylabel('累计收益 (%)')
        ax.axhline(0, color='black', linewidth=0.5)
        plt.tight_layout()
        plt.show()
        
        # Exit reason 分布
        print('\nExit Reason 分布:')
        print(rdf['exit_reason'].value_counts())
        
        # Label analysis
        if detail.get('label_analysis'):
            print('\nTriple Barrier Label Analysis:')
            for k, v in detail['label_analysis'].items():
                print(f'  {k}: {v}')

## Step 7: Regime 条件回测

测试指标在不同市场状态下的表现差异。

In [ ]:
# 计算 regime 指标
rv = realized_volatility(prices_1m, window=60)
vol_reg = volatility_regime(rv, lookback=500)

# ADX (需要 OHLCV)
adx_df = adx_indicator(ohlcv_1m.loc[common_idx])

# Amihud
amihud = amihud_illiquidity(ohlcv_1m.loc[common_idx])

regime_df = pd.DataFrame({
    'vol_regime': vol_reg.loc[common_idx] if common_idx[0] in vol_reg.index else 'unknown',
    'adx': adx_df.loc[common_idx, 'adx'] if not adx_df.empty else 0,
    'amihud': amihud.loc[common_idx] if common_idx[0] in amihud.index else 0,
}, index=common_idx)

print('Volatility Regime 分布:')
print(regime_df['vol_regime'].value_counts())

In [ ]:
# 在不同 vol regime 下的指标表现
if not bt_summary.empty:
    best = bt_summary['sharpe'].idxmax()
    config = indicator_configs.get(best, {'long_threshold': 0, 'short_threshold': 0})
    
    for regime_val in ['low', 'medium', 'high']:
        mask = regime_df['vol_regime'] == regime_val
        regime_idx = regime_df[mask].index
        
        if len(regime_idx) < 100:
            print(f'{regime_val:>7s} regime: insufficient data ({len(regime_idx)} bars)')
            continue
        
        ind_sub = indicators_1m.loc[indicators_1m.index.isin(regime_idx), best].dropna()
        price_sub = prices_1m.loc[prices_1m.index.isin(regime_idx)]
        
        r = backtest_single_indicator(ind_sub, price_sub, **config, **BT_PARAMS)
        m = r['metrics']
        print(f'{regime_val:>7s} regime: trades={m["n_signals"]:>4d}  '
              f'win={m["win_rate"]:>5.1f}%  sharpe={m["sharpe"]:>6.2f}')

## Step 8: 保存结果

In [ ]:
# 保存回测汇总
from utilities.paths import DataPaths
save_path = DataPaths.backtest_result('indicator_research', 'single_indicator_summary')
bt_summary.to_csv(save_path)
print(f'Saved to: {save_path}')

# 保存特征矩阵（供后续模型训练）
feat_path = DataPaths.features_dir(SYMBOL, 'base_indicators')
indicators_1m.to_parquet(feat_path / 'all_indicators_1m.parquet')
print(f'Features saved to: {feat_path}')